# 01 — Python para matemáticas

## Motivación

**numpy** es el idioma numérico de este curso: vectores, matrices, funciones y
operaciones estadísticas se escriben casi igual que en la notación matemática que ya
conoces, pero sin ciclos `for` explícitos.

Este notebook funciona como referencia permanente durante el curso: vuelve a él cuando
un fragmento de código no resulte claro.

**Contenido:**
1. Arrays
2. Indexado y máscaras
3. Vectorización
4. Álgebra lineal básica
5. Estadística y agregaciones
6. Aleatoriedad reproducible
7. Graficación con Plotly

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 1. Arrays

Un vector $\mathbf{x} = (x_1, \ldots, x_n) \in \mathbb{R}^n$ es un `np.array`.
Algunas formas de construirlos:

- Valores explícitos: `np.array([...])`
- La secuencia $0, 1, \ldots, n$: `np.arange(n)`
- Una partición uniforme de $[a, b]$ en $k$ puntos: `np.linspace(a, b, k)`
- Vectores constantes: `np.zeros(n)`, `np.ones(n)`

In [ ]:
x = np.array([1, 2, 3, 4, 5])
n = np.arange(0, 11)
t = np.linspace(0, 10, 5)

print(f"x = {x}")
print(f"n = {n}")
print(f"t = {t}")
print(f"\nshape de x: {x.shape}   dtype de x: {x.dtype}   len(x): {len(x)}")
print(f"zeros(3): {np.zeros(3)}   ones(3): {np.ones(3)}")

## 2. Indexado y máscaras

$x_i$ se escribe `x[i]` — con una diferencia respecto a la notación matemática habitual:
**los índices empiezan en 0**. `x[0]` es el primer elemento.

Además de índices individuales, numpy permite **rebanadas** (`x[2:5]`) y, lo más útil
para este curso, **máscaras booleanas**: el subconjunto $\{x_i : x_i > 0\}$ se escribe
`x[x > 0]`. Filtrar datos por una condición, sin ciclos, es algo que harás constantemente.

In [ ]:
x = np.array([4, -2, 7, 0, -5, 9, 3])

print(f"x[0]   = {x[0]}   (primer elemento, no x[1])")
print(f"x[-1]  = {x[-1]}  (último elemento)")
print(f"x[2:5] = {x[2:5]} (rebanada)")

mask = x > 0
print(f"\nmáscara x > 0: {mask}")
print(f"x[x > 0]: {x[mask]}")
print(f"cuántos positivos: {np.sum(mask)}")

## 3. Vectorización

Si $y_i = f(x_i)$ para cada elemento, numpy lo escribe **sin ciclos**: la operación se
aplica a todo el array de una vez. La regla del curso: *si estás escribiendo un `for`
para hacer matemáticas sobre un array, probablemente hay una forma vectorizada.*

Las funciones matemáticas de numpy (`sin`, `exp`, `log`, `sqrt`, ...) ya vienen
vectorizadas — actúan elemento a elemento automáticamente.

Para comparar ambos enfoques, calculamos $\sum_{i=1}^{n} x_i^2$ de dos formas: con un
ciclo `for` explícito y con `np.sum(x**2)`.

In [ ]:
x = np.array([1, 4, 9, 16])

print(f"x + 10:    {x + 10}")
print(f"x * 2:     {x * 2}")
print(f"sqrt(x):   {np.sqrt(x)}")
print(f"exp(x/10): {np.exp(x / 10)}")

In [ ]:
import time

grande = np.arange(1_000_000)

inicio = time.perf_counter()
total_ciclo = 0
for valor in grande:
    total_ciclo += valor**2
tiempo_ciclo = time.perf_counter() - inicio

inicio = time.perf_counter()
total_vectorizado = np.sum(grande**2)
tiempo_vectorizado = time.perf_counter() - inicio

print(f"con ciclo for:    {tiempo_ciclo:.4f} s")
print(f"vectorizado:      {tiempo_vectorizado:.4f} s")
print(f"factor:           {tiempo_ciclo / tiempo_vectorizado:.0f}x más rápido")
print(f"mismo resultado:  {total_ciclo == total_vectorizado}")

El mismo principio — operar sobre el array completo, sin ciclos — resuelve una tarea
que aparece todo el curso: **centrar las columnas de una matriz** (restar a cada
columna su propio promedio). Con $X \in \mathbb{R}^{n \times d}$ ($n$ filas, $d$ columnas):

$$X'_{ij} = X_{ij} - \bar{x}_j, \qquad \bar{x}_j = \frac{1}{n}\sum_{i=1}^{n} X_{ij}$$

numpy lo hace en una línea:

```python
X_centrada = X - X.mean(axis=0)
```

numpy "expande" automáticamente el vector de promedios $\bar{\mathbf{x}} = (\bar{x}_1,
\ldots, \bar{x}_d)$ para restarlo de cada fila de $X$ — a esto se le llama
*broadcasting*. Lo verás en acción en la sección 4.

## 4. Álgebra lineal básica

Una matriz $A \in \mathbb{R}^{m \times n}$ es un array 2D con `shape (m, n)`.

- Producto matriz-vector $A\mathbf{v}$ → `A @ v`. **Advertencia**: `@` es producto
  matricial; `*` es producto elemento a elemento — confundirlos es el error más común
  al empezar con numpy.
- Transpuesta $A^\top$ → `A.T`
- Sistema lineal $A\mathbf{x} = \mathbf{b}$ → `np.linalg.solve(A, b)`

No derivamos aquí *cómo* funciona `solve` — por ahora es una herramienta. Lo usarás con
propósito, y con más contexto, en la sesión 02.

In [ ]:
A = np.array([[2.0, 1.0, 0.0], [1.0, 3.0, 1.0], [0.0, 1.0, 2.0]])
v = np.array([1.0, 2.0, 3.0])

print(f"A:\n{A}")
print(f"\nA @ v  (producto matricial): {A @ v}")
print(f"A * v  (elemento a elemento, ¡NO es lo mismo!):\n{A * v}")
print(f"\nA.T (transpuesta):\n{A.T}")

b = np.array([3.0, 6.0, 5.0])
x_sol = np.linalg.solve(A, b)
print(f"\nsolución de A x = b: {x_sol}")
print(f"verificación A @ x ≈ b: {np.allclose(A @ x_sol, b)}")

## 5. Estadística y agregaciones

$\bar{x}$, $\sigma$, $\max$, $\arg\max$ → `mean`, `std`, `max`, `argmax`.

Sobre una matriz, el parámetro `axis` dice qué eje colapsa la agregación:
`axis=0` reduce filas (da un resultado por **columna**); `axis=1` reduce columnas
(da un resultado por **fila**).

In [ ]:
x = np.array([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0])
print(f"media: {x.mean():.3f}   std: {x.std():.3f}   max: {x.max()}   argmax: {x.argmax()}")

M = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])  # shape (2, 3)
print(f"\nM:\n{M}")
print(f"media por columna (axis=0): {M.mean(axis=0)}")
print(f"media por fila    (axis=1): {M.mean(axis=1)}")

## 6. Aleatoriedad reproducible

ML recurre a la aleatoriedad de forma constante: dividir train/test, generar ruido,
inicializar parámetros. Para que un experimento sea **reproducible** — que distintas
ejecuciones produzcan el mismo resultado — se fija una **semilla**.

El curso usa el API moderno de numpy: un generador explícito,
`rng = np.random.default_rng(seed)`.

In [ ]:
rng_a = np.random.default_rng(seed=42)
rng_b = np.random.default_rng(seed=42)

print(f"rng_a.uniform:  {rng_a.uniform(0, 1, size=3)}")
print(f"rng_b.uniform:  {rng_b.uniform(0, 1, size=3)}   (misma semilla → misma secuencia)")
print(f"rng_a.normal:   {rng_a.normal(0, 1, size=3)}")
print(f"rng_a.integers: {rng_a.integers(0, 10, size=3)}")

## 7. Graficación con Plotly

En Plotly, una gráfica es un objeto `go.Figure` al que se le agregan **trazas**
(`go.Scatter`, `go.Histogram`, ...) y luego se configura con `update_layout`. A
diferencia de una imagen estática, es interactiva: puedes hacer zoom, desplazarte y
pasar el cursor sobre los puntos para ver sus valores.

Regla del curso: los ejes siempre van etiquetados, con unidades cuando existan.

Empezamos graficando $y(t) = \sin(t)$ para $t \in [0, 2\pi]$.

In [ ]:
t = np.linspace(0, 2 * np.pi, 100)
y = np.sin(t)

fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=y, mode="lines", name="sin(t)"))
fig.update_layout(
    title="Función sinusoidal",
    xaxis_title="t [s]",
    yaxis_title="amplitud",
    template="plotly_white",
)
fig.show()

Ahora dos trazas en la misma figura: $y(t) = \sin(t)$ y $y(t) = \cos(t)$.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=np.sin(t), mode="lines", name="sin(t)"))
fig.add_trace(go.Scatter(x=t, y=np.cos(t), mode="lines", name="cos(t)", line=dict(dash="dash")))
fig.update_layout(
    title="Seno y coseno",
    xaxis_title="t [s]",
    yaxis_title="amplitud",
    template="plotly_white",
    hovermode="x unified",
)
fig.show()

Un vistazo a lo que viene en la sesión 02: datos generados con

$$y = 2x + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, 1)$$

y la distribución de ese ruido $\varepsilon$.

In [ ]:
rng = np.random.default_rng(seed=0)
x_datos = rng.uniform(0, 10, size=200)
ruido = rng.normal(0, 1, size=200)
y_datos = 2 * x_datos + ruido

fig = make_subplots(rows=1, cols=2, subplot_titles=("y = 2x + ruido", "distribución del ruido"))

fig.add_trace(
    go.Scatter(x=x_datos, y=y_datos, mode="markers", marker=dict(size=5, opacity=0.5), showlegend=False),
    row=1, col=1,
)
fig.add_trace(
    go.Histogram(x=ruido, nbinsx=30, showlegend=False),
    row=1, col=2,
)

fig.update_xaxes(title_text="x", row=1, col=1)
fig.update_yaxes(title_text="y", row=1, col=1)
fig.update_xaxes(title_text="ε", row=1, col=2)
fig.update_yaxes(title_text="frecuencia", row=1, col=2)

fig.update_layout(template="plotly_white", height=400)
fig.show()

## Tabla de equivalencias: notación matemática → numpy

| Notación | numpy |
|---|---|
| $\sum_i x_i$ | `np.sum(x)` |
| $\mathbf{u} \cdot \mathbf{v}$ | `u @ v` |
| $A\mathbf{v}$ | `A @ v` |
| $A^\top$ | `A.T` |
| $A^{-1}\mathbf{b}$ | `np.linalg.solve(A, b)` (nunca `np.linalg.inv`) |
| $\bar{x}$ | `x.mean()` |
| $\sigma$ | `x.std()` |
| $x_i$ | `x[i]` — índices desde **0** |
| $\{x_i : P(x_i)\}$ | `x[condición]` (máscara booleana) |

## Ejercicio

Trabaja en una copia de este notebook dentro de `mi-trabajo/`.

1. **Ecuaciones normales.** Genera datos sintéticos con $y = 3x - 2 + \varepsilon$
   usando `rng.normal` para $\varepsilon$. Construye la matriz de diseño
   $X = [\mathbf{1} \mid \mathbf{x}]$ con `np.column_stack` y resuelve
   $X^\top X \mathbf{w} = X^\top \mathbf{y}$ con `np.linalg.solve`. Verifica que
   recuperas aproximadamente $(-2, 3)$. *(Esto es exactamente lo que harás, con más
   contexto, en la sesión 02.)*

2. **Máscaras.** Genera 50 calificaciones con `rng.uniform(4, 10, 50)`. Sin usar
   ningún ciclo, calcula: cuántas aprueban (≥ 6), el promedio de las aprobadas, y la
   calificación máxima entre las reprobadas.

3. **`axis` y Plotly.** Construye una matriz 2D de temperaturas simuladas con `rng.normal`
   (filas = 30 días, columnas = 3 ciudades). Calcula el promedio por ciudad y el
   promedio por día usando `axis`. Grafica con Plotly una traza de línea por ciudad
   (el eje x son los días).